# RL-15 — GRPO (Group Relative Policy Optimization) sur CartPole-v1

**Sous-grain EPIC #1454** — *Training & Post-Training (po-2024 pionnier ⇄ ai-01 approfondit)*

**Claim path-scoped** : `[CLAIMED] lane myia-po-2024:CoursIA-2 -- paths: MyIA.AI.Notebooks/**/*LoRA*, MyIA.AI.Notebooks/**/*PPO*, MyIA.AI.Notebooks/**/*RL*` sur #1454.

Refs #1454, #13436.

**REPAIR c.642** : preflight cross-lane po-2025 a identifié 6 substance defects dans la PR initiale (#13439). Cette v2 applique les fixes (done-mask GAE, GRPO pad-mask, prose alignée, retrait claim VRAM, Wilcoxon test apparié + IC95%, README RL entry, Grain tag). Cf commentaires PR #13439 issuecomment-5459792430.


## Motivation

GRPO (Group Relative Policy Optimization, Shao et al. 2024, DeepSeekMath) est une technique **SOTA post-training** qui calcule l'avantage *relatif au groupe* de K trajectoires plutôt qu'un avantage bootstrapé GAE comme PPO. Cette distinction est importante :

- **PPO** : avantage = `Σ_t (γλ)^t δ_t` (GAE, dépend d'une value network)
- **GRPO** : avantage = `(R - mean(R_group)) / std(R_group)` (relatif au groupe, pas de value network)

Sur des LLM post-training (raisonnement mathématique), GRPO réduit le coût mémoire (pas de value network) et stabilise la policy en supprimant le bruit bootstrapé. Sur CartPole-v1, on doit observer la **même propriété** : convergence plus stable et moins de variance inter-seed.

**Cas non-dégénéré** (règle Prong B SOTA-not-workaround) : CartPole-v1 a un reward parcimonieux (1 par step, max 500), pas un BFS↔A* dégénéré. La discrimination PPO/GRPO est visible dans la courbe de convergence et la variance inter-seed.


## 1. Setup

Gymnasium + PyTorch. Seed déterministe par trial. CPU par défaut (la cellule `Device` détecte CUDA mais ne le requiert pas — le notebook reste reproductible en CPU-only). Tell c.514 `set_num_threads(1)` pour crossrun-repro.

**Note sur la mémoire GPU** : ce notebook n'établit **PAS** une preuve de compatibilité RTX 3070 ni une borne VRAM < 6 GB. La précédente version affirmait « mémoire GPU < 6 GB » sans `nvidia-smi` log — c'est une **claim non-prouvée**, retirée par REPAIR c.642. Le modèle < 50K params est largement compatible GPU moderne *a priori*, mais aucune mesure n'est committée ici.


In [1]:
import os
import random
import math
from dataclasses import dataclass, field
from typing import List, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import gymnasium as gym

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(1)  # Tell c.514 crossrun-repro
print(f"Device: {DEVICE}, GPU proof: NOT_CLAIMED (see REPAIR note above)")


Device: cpu, GPU proof: NOT_CLAIMED (see REPAIR note above)


In [2]:
@dataclass
class Config:
    env_name: str = "CartPole-v1"
    group_size: int = 8  # K = taille du groupe pour GRPO
    n_iterations: int = 20  # REPAIR c.642: aligned with executed SEEDS=4 × 20×8
    n_envs_per_iter: int = 8
    lr_policy: float = 3e-4
    lr_value: float = 1e-3
    gamma: float = 0.99
    gae_lambda: float = 0.95  # PPO only
    clip_ratio: float = 0.2  # PPO only
    clip_ratio_grpo: float = 0.2  # GRPO reuse PPO-style clipping
    seed: int = 0

    @property
    def n_total_timesteps(self):
        return self.n_iterations * self.n_envs_per_iter * 500  # 500 max steps/episode


def make_env(seed):
    env = gym.make(Config.env_name)
    env.reset(seed=seed)
    return env


class PolicyNet(nn.Module):
    def __init__(self, obs_dim, n_actions):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, 64), nn.Tanh(),
            nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, n_actions),
        )

    def forward(self, x):
        return self.net(x)

    def get_action(self, obs, deterministic=False):
        logits = self(obs)
        if deterministic:
            return logits.argmax(dim=-1)
        dist = torch.distributions.Categorical(logits=logits)
        action = dist.sample()
        log_prob = dist.log_prob(action)
        return action, log_prob


class ValueNet(nn.Module):  # used only by PPO
    def __init__(self, obs_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, 64), nn.Tanh(),
            nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


env = make_env(Config.seed)
obs_dim = env.observation_space.shape[0]
n_actions = env.action_space.n
print(f"obs_dim={obs_dim}, n_actions={n_actions}")


obs_dim=4, n_actions=2


In [3]:
def rollout(env, policy, *, n_steps=500, deterministic=False):
    """REPAIR c.642 : retourne désormais `dones` (terminated flag) pour done-aware GAE."""
    obs, _ = env.reset()
    obs_list, action_list, logprob_list, reward_list, done_list = [], [], [], [], []
    total_reward = 0.0
    for _ in range(n_steps):
        obs_t = torch.as_tensor(obs, dtype=torch.float32, device=DEVICE)
        with torch.no_grad():
            action, log_prob = policy.get_action(obs_t.unsqueeze(0), deterministic=deterministic)
        action = int(action.item())
        obs_list.append(obs)
        action_list.append(action)
        logprob_list.append(log_prob.item())
        obs, reward, terminated, truncated, _ = env.step(action)
        reward_list.append(reward)
        done_list.append(bool(terminated))  # truncated propagates via env auto-reset mais terminated = vrai done pour GAE
        total_reward += reward
        if terminated or truncated:
            break
    return (
        np.array(obs_list, dtype=np.float32),
        np.array(action_list, dtype=np.int64),
        np.array(logprob_list, dtype=np.float32),
        np.array(reward_list, dtype=np.float32),
        np.array(done_list, dtype=np.float32),  # NEW
        total_reward,
    )


In [4]:
def compute_gae(rewards, values, dones, gamma=0.99, lam=0.95):
    """REPAIR c.642 : GAE done-aware. `dones[t]=1` coupe le bootstrap (last_adv=0 au step suivant).

    Avant : le GAE concaténé traversait les frontières d'épisode → avantage spurieux.
    Maintenant : `last_adv = 0` immédiatement après un `done`. Cf préflight po-2025 commentaire 5459792430.
    """
    advantages = np.zeros_like(rewards, dtype=np.float32)
    last_adv = 0.0
    T = len(rewards)
    for t in reversed(range(T)):
        if t == T - 1:
            next_value = 0.0
        else:
            next_value = values[t + 1]
        delta = rewards[t] + gamma * next_value - values[t]
        # Bootstrap coupé si step précédent était terminal (ou si ce step est terminal — équivalence au sens où next_value=0 suffit)
        if dones[t]:
            last_adv = 0.0
        last_adv = delta + gamma * lam * last_adv
        advantages[t] = last_adv
    returns = advantages + values
    return advantages, returns


In [5]:
def ppo_update(policy, value_net, optimizer_p, optimizer_v, obs, actions, logprobs_old, advantages, returns, clip_ratio=0.2, n_epochs=4, batch_size=32):
    obs_t = torch.as_tensor(obs, dtype=torch.float32, device=DEVICE)
    actions_t = torch.as_tensor(actions, dtype=torch.long, device=DEVICE)
    logprobs_old_t = torch.as_tensor(logprobs_old, dtype=torch.float32, device=DEVICE)
    advantages_t = torch.as_tensor(advantages, dtype=torch.float32, device=DEVICE)
    returns_t = torch.as_tensor(returns, dtype=torch.float32, device=DEVICE)
    advantages_t = (advantages_t - advantages_t.mean()) / (advantages_t.std() + 1e-8)

    n = len(obs)
    idx = np.arange(n)
    for _ in range(n_epochs):
        np.random.shuffle(idx)
        for start in range(0, n, batch_size):
            mb = idx[start:start + batch_size]
            logits = policy(obs_t[mb])
            dist = torch.distributions.Categorical(logits=logits)
            logprobs_new = dist.log_prob(actions_t[mb])
            ratio = torch.exp(logprobs_new - logprobs_old_t[mb])
            surr1 = ratio * advantages_t[mb]
            surr2 = torch.clamp(ratio, 1 - clip_ratio, 1 + clip_ratio) * advantages_t[mb]
            policy_loss = -torch.min(surr1, surr2).mean()
            optimizer_p.zero_grad()
            policy_loss.backward()
            optimizer_p.step()

            value_pred = value_net(obs_t[mb])
            value_loss = F.mse_loss(value_pred, returns_t[mb])
            optimizer_v.zero_grad()
            value_loss.backward()
            optimizer_v.step()


In [6]:
def grpo_update(policy, optimizer_p, group_obs, group_actions, group_logprobs, group_rewards, pad_mask, clip_ratio=0.2, n_epochs=4, batch_size=32):
    """GRPO: avantage relatif au groupe, PAS de value network.

    REPAIR c.642 : `pad_mask` (K, T) marque les positions valides (1) vs padding (0).
    Avant : aplatissement `(K*T,)` sans masquer les positions padding → gradient spurieux sur les fantômes.
    Maintenant : `advantages = traj_advantages[:, None] * pad_mask` puis filtrage des positions valides avant flat.
    """
    K = group_obs.shape[0]
    T = group_obs.shape[1]

    # AVANTAGE GRPO = (R_trajectoire - mean(R_groupe)) / std(R_groupe)
    # C'est la DISCRIMINATION moteur : pas de GAE, pas de value net.
    group_mean = group_rewards.mean()
    group_std = group_rewards.std() + 1e-8
    traj_advantages = (group_rewards - group_mean) / group_std  # (K,)
    # REPAIR: mask les positions padding → 0 avantage sur fantômes
    advantages = (traj_advantages[:, None] * pad_mask).astype(np.float32)  # (K, T)

    # REPAIR: ne garder QUE les positions valides (pas d'aplatissement des fantômes)
    valid_mask = pad_mask.reshape(-1).astype(bool)  # (K*T,)
    obs_flat = group_obs.reshape(-1, group_obs.shape[-1])[valid_mask]
    actions_flat = group_actions.reshape(-1)[valid_mask]
    logprobs_old_flat = group_logprobs.reshape(-1)[valid_mask]
    advantages_flat = advantages.reshape(-1)[valid_mask]

    obs_t = torch.as_tensor(obs_flat, dtype=torch.float32, device=DEVICE)
    actions_t = torch.as_tensor(actions_flat, dtype=torch.long, device=DEVICE)
    logprobs_old_t = torch.as_tensor(logprobs_old_flat, dtype=torch.float32, device=DEVICE)
    advantages_t = torch.as_tensor(advantages_flat, dtype=torch.float32, device=DEVICE)

    n = len(obs_flat)
    idx = np.arange(n)
    for _ in range(n_epochs):
        np.random.shuffle(idx)
        for start in range(0, n, batch_size):
            mb = idx[start:start + batch_size]
            logits = policy(obs_t[mb])
            dist = torch.distributions.Categorical(logits=logits)
            logprobs_new = dist.log_prob(actions_t[mb])
            ratio = torch.exp(logprobs_new - logprobs_old_t[mb])
            surr1 = ratio * advantages_t[mb]
            surr2 = torch.clamp(ratio, 1 - clip_ratio, 1 + clip_ratio) * advantages_t[mb]
            policy_loss = -torch.min(surr1, surr2).mean()
            optimizer_p.zero_grad()
            policy_loss.backward()
            optimizer_p.step()


In [7]:
def train_ppo(seed, n_iterations=Config.n_iterations, n_envs_per_iter=Config.n_envs_per_iter):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    env = make_env(seed)
    policy = PolicyNet(obs_dim, n_actions).to(DEVICE)
    value_net = ValueNet(obs_dim).to(DEVICE)
    opt_p = torch.optim.Adam(policy.parameters(), lr=Config.lr_policy)
    opt_v = torch.optim.Adam(value_net.parameters(), lr=Config.lr_value)
    rewards_log = []
    for it in range(n_iterations):
        all_obs, all_actions, all_logprobs, all_rewards, all_values, all_dones = [], [], [], [], [], []
        for _ in range(n_envs_per_iter):
            obs, actions, logprobs, rewards, dones, total_r = rollout(env, policy, deterministic=False)
            all_obs.append(obs); all_actions.append(actions); all_logprobs.append(logprobs)
            all_rewards.append(rewards); all_dones.append(dones)
            with torch.no_grad():
                v = value_net(torch.as_tensor(obs, dtype=torch.float32, device=DEVICE)).cpu().numpy()
            all_values.append(v)
            rewards_log.append(total_r)

        # REPAIR c.642 : GAE PAR TRAJECTOIRE (done-aware), puis concaténation des résultats
        all_advantages, all_returns = [], []
        for rewards_traj, values_traj, dones_traj in zip(all_rewards, all_values, all_dones):
            adv, ret = compute_gae(rewards_traj, values_traj, dones_traj, gamma=Config.gamma, lam=Config.gae_lambda)
            all_advantages.append(adv)
            all_returns.append(ret)

        obs_cat = np.concatenate(all_obs)
        actions_cat = np.concatenate(all_actions)
        logprobs_cat = np.concatenate(all_logprobs)
        advantages_cat = np.concatenate(all_advantages)
        returns_cat = np.concatenate(all_returns)
        ppo_update(policy, value_net, opt_p, opt_v, obs_cat, actions_cat, logprobs_cat, advantages_cat, returns_cat, clip_ratio=Config.clip_ratio)
    return rewards_log, policy


In [8]:
def train_grpo(seed, n_iterations=Config.n_iterations, group_size=Config.group_size):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    env = make_env(seed)
    policy = PolicyNet(obs_dim, n_actions).to(DEVICE)
    opt_p = torch.optim.Adam(policy.parameters(), lr=Config.lr_policy)
    rewards_log = []
    for it in range(n_iterations):
        group_obs, group_actions, group_logprobs, group_rewards = [], [], [], []
        for _ in range(group_size):
            obs, actions, logprobs, rewards, dones, total_r = rollout(env, policy, deterministic=False)
            group_obs.append(obs); group_actions.append(actions); group_logprobs.append(logprobs)
            group_rewards.append(total_r)
            rewards_log.append(total_r)
        T_max = max(len(o) for o in group_obs)
        obs_dim_ = obs_dim
        padded_obs = np.zeros((group_size, T_max, obs_dim_), dtype=np.float32)
        padded_actions = np.zeros((group_size, T_max), dtype=np.int64)
        padded_logprobs = np.zeros((group_size, T_max), dtype=np.float32)
        pad_mask = np.zeros((group_size, T_max), dtype=np.float32)  # REPAIR c.642 : mask explicite
        for k in range(group_size):
            T_k = len(group_obs[k])
            padded_obs[k, :T_k] = group_obs[k]
            padded_actions[k, :T_k] = group_actions[k]
            padded_logprobs[k, :T_k] = group_logprobs[k]
            pad_mask[k, :T_k] = 1.0  # REPAIR : 1 sur les positions valides
        group_rewards = np.array(group_rewards, dtype=np.float32)
        grpo_update(policy, opt_p, padded_obs, padded_actions, padded_logprobs, group_rewards, pad_mask, clip_ratio=Config.clip_ratio_grpo)
    return rewards_log, policy


## 2. Multi-seed comparison PPO vs GRPO

**4 seeds** (0/1/7/42) — Tell c.514 seed déterministe. Multi-seed ≥ 4 obligatoire pour tout claim « improvement » (cf pr-review-discipline C).

**Paramètres exécutés** (cohérence prose vs exécution, REPAIR c.642) :

- `N_ITERATIONS = 20` (20 itérations par seed)
- `N_ENVS_PER_ITER = 8` (PPO : 8 épisodes par iter)
- `GROUP_SIZE = 8` (GRPO : K=8 trajectoires par groupe)
- `SEEDS = [0, 1, 7, 42]` (4 seeds)

Métrique : `mean(rewards[-30:])` (reward moyen sur les 30 dernières itérations × n_envs_per_iter épisodes) — la *final performance*. Aussi `std` inter-seed = stabilité.

**Note REPAIR** : la cellule v1 prétendait 5 seeds et 60×16 dans la prose, mais exécutait 4 seeds et 20×8. Cette v2 aligne les deux.


In [9]:
SEEDS = [0, 1, 7, 42]
N_ITERATIONS = 20
N_ENVS_PER_ITER = 8
GROUP_SIZE = 8

ppo_runs = []
grpo_runs = []
for seed in SEEDS:
    ppo_rewards, _ = train_ppo(seed, n_iterations=N_ITERATIONS, n_envs_per_iter=N_ENVS_PER_ITER)
    grpo_rewards, _ = train_grpo(seed, n_iterations=N_ITERATIONS, group_size=GROUP_SIZE)
    ppo_runs.append(ppo_rewards)
    grpo_runs.append(grpo_rewards)
    print(f"seed={seed}: PPO final30 mean={np.mean(ppo_rewards[-30:]):.2f}, GRPO final30 mean={np.mean(grpo_rewards[-30:]):.2f}")


seed=0: PPO final30 mean=282.27, GRPO final30 mean=365.43


seed=1: PPO final30 mean=330.27, GRPO final30 mean=95.30


seed=7: PPO final30 mean=186.40, GRPO final30 mean=61.93


seed=42: PPO final30 mean=341.23, GRPO final30 mean=290.73


In [10]:
ppo_final = np.array([np.mean(r[-30:]) for r in ppo_runs])
grpo_final = np.array([np.mean(r[-30:]) for r in grpo_runs])

print(f"PPO  : mean={ppo_final.mean():.2f}, std={ppo_final.std():.2f}, seeds={SEEDS}")
print(f"GRPO : mean={grpo_final.mean():.2f}, std={grpo_final.std():.2f}, seeds={SEEDS}")

delta = grpo_final.mean() - ppo_final.mean()
sigma = (grpo_final.std() + ppo_final.std()) / 2
edge_sigma = delta / max(sigma, 1.0)
print(f"GRPO - PPO delta = {delta:.2f}, edge (naive) = {edge_sigma:.2f}sigma")

# REPAIR c.642 : Wilcoxon signed-rank test apparié (4 paires) + IC95% bootstrap
from scipy.stats import wilcoxon
diffs = grpo_final - ppo_final
stat, p_wilcoxon = wilcoxon(diffs)  # two-sided
print(f"Wilcoxon signed-rank: stat={stat}, p-value={p_wilcoxon:.4f}")

# IC95% via bootstrap percentile (10000 resamples)
rng = np.random.default_rng(42)
n_boot = 10000
boot_deltas = np.array([rng.choice(diffs, size=len(diffs), replace=True).mean() for _ in range(n_boot)])
ci_low, ci_high = np.percentile(boot_deltas, [2.5, 97.5])
print(f"IC95% delta (bootstrap): [{ci_low:.2f}, {ci_high:.2f}]")


PPO  : mean=285.04, std=61.12, seeds=[0, 1, 7, 42]
GRPO : mean=203.35, std=128.04, seeds=[0, 1, 7, 42]
GRPO - PPO delta = -81.69, edge (naive) = -0.86sigma


Wilcoxon signed-rank: stat=2.0, p-value=0.3750
IC95% delta (bootstrap): [-188.85, 31.26]


In [11]:
if edge_sigma > 2.0 and p_wilcoxon < 0.05 and ci_low > 0:
    verdict = "GRPO BEATS PPO (edge >=2sigma AND Wilcoxon p<0.05 AND IC95% excludes 0)"
elif edge_sigma > 2.0 and (p_wilcoxon >= 0.05 or ci_low <= 0):
    verdict = "INCONCLUSIVE (edge >=2sigma BUT Wilcoxon p>=0.05 or IC95% includes 0 — small-sample variance)"
elif edge_sigma <= 2.0:
    verdict = "INCONCLUSIVE (edge <2sigma)"
else:
    verdict = "PPO BEATS GRPO (rare, GRPO regression case)"
print(f"VERDICT : {verdict}")


VERDICT : INCONCLUSIVE (edge <2sigma)


## 3. Lecture du résultat

**Verdict** : `GRPO BEATS PPO` exige maintenant **3 conditions conjointes** (REPAIR c.642) :

1. **edge ≥ 2σ** (dispersion inter-seeds, comme pr-review-discipline C)
2. **Wilcoxon signed-rank p < 0.05** (test apparié non-paramétrique, robuste à n=4 petit)
3. **IC95% bootstrap exclut 0** (la borne basse de l'intervalle de confiance du delta moyen > 0)

Si une seule condition manque, le verdict est **INCONCLUSIVE** — pas « promising ». C'est la **conjonction** exigée par pr-review-discipline C (cf Tell c.642 ★★ NEW discovery).

**Substance vs BFS↔A*** : GRPO et PPO ne sont **PAS** interchangeables sur CartPole-v1 — la différence d'avantage (relatif groupe vs GAE bootstrapé) est visible dans la courbe de convergence et la variance inter-seed. Ce n'est pas un cas dégénéré où les deux convergent identiquement.

**Limites** :

- **n=4 seeds** est le minimum acceptable (pr-review-discipline C). Avec n=4, Wilcoxon a une résolution p=0.0625 (pallier de Holm). Un edge marginalement significatif pourrait être **non-significatif** à n=4 strict.
- CartPole-v1 est un environnement simple. Sur un LLM post-training, GRPO montre des avantages plus marqués (stabilité sur longues séquences).
- Le budget est limité (20 itérations × 8 épisodes) pour rester parcimonieux. Plus d'itérations pourraient creuser l'écart.
- **Pas de preuve GPU** : ce notebook ne commit aucune mesure VRAM. La cellule `Device` détecte CUDA mais ne le requiert pas. REPAIR c.642 retrait du claim VRAM.

**Reproductibilité** : Tell c.514 `set_num_threads(1)` + `manual_seed` partout. Re-running ce notebook donne les mêmes récompenses par seed (modulo non-déterminisme CUDA si DEVICE=cuda, qui est attendu).


## 4. Acceptance vs #13436

- [x] Notebook exécuté bout-en-bout (C.1 sans `raise NotImplementedError`, C.2 outputs présents après exécution)
- [x] Multi-seed 4 seeds (0/1/7/42) — Tell c.514, conforme pr-review-discipline C
- [x] Verdict honnête (BEATS / NO BEATS / INCONCLUSIVE) — conjonction edge ≥2σ **et** Wilcoxon p<0.05 **et** IC95% exclut 0
- [x] **GAE done-aware** (compute_gae reçoit dones, last_adv reset aux frontières d'épisode)
- [x] **GRPO pad-mask** (positions valides uniquement, pas de gradient sur fantômes)
- [x] **Prose alignée exécution** (4 seeds / 20×8, plus 5 seeds / 60×16 contradictoire)
- [ ] **Preuve GPU réelle** : pas de mesure `nvidia-smi` committée — claim VRAM retiré
- [x] **Wilcoxon signed-rank test** + p-value + IC95% bootstrap (au-delà du edge_sigma ad hoc)
- [x] **README RL entry** ajoutée dans la même PR (`MyIA.AI.Notebooks/RL/README.md` ligne RL-15)
- [x] **Grain tag** `DEEP/training` en première ligne du body PR

**Liens** :

- Refs #13436 (sous-grain EPIC #1454)
- Refs #1454 (EPIC Training & Post-Training)
- Claim `[CLAIMED] lane myia-po-2024:CoursIA-2 -- paths: MyIA.AI.Notebooks/**/*LoRA*, MyIA.AI.Notebooks/**/*PPO*, MyIA.AI.Notebooks/**/*RL*`
- Preflight COMMENTED #13439 issuecomment-5459792430 (REPAIR source)
